# Feature Engineering Experiment


## 1. Data loading and merge

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")
features = pd.read_csv(f"{DATA_DIR}/features.csv")

train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])
features["Date"] = pd.to_datetime(features["Date"])

In [2]:
train_merged = train.merge(stores, on="Store", how="left")
train_merged = train_merged.merge(features, on=["Store", "Date", "IsHoliday"], how="left")

test_merged = test.merge(stores, on="Store", how="left")
test_merged = test_merged.merge(features, on=["Store", "Date", "IsHoliday"], how="left")

assert len(train_merged) == len(train)
assert len(test_merged) == len(test)

print("Train merged shape:", train_merged.shape)
print("Test merged shape:", test_merged.shape)

Train merged shape: (421570, 16)
Test merged shape: (115064, 15)


## 2. Missing Values

### 2.1 CPI / Unemployment forward-fill for each Store
last 13 test weeks are missing CPI/Unemployment for all stores.ffill in each store. these are slow moving economic indicators.

In [3]:
train_merged = train_merged.sort_values(["Store", "Date"])
test_merged = test_merged.sort_values(["Store", "Date"])

for col in ["CPI", "Unemployment"]:
    train_merged[col] = train_merged.groupby("Store")[col].ffill()
    test_merged[col] = test_merged.groupby("Store")[col].ffill()

### 2.2 MarkDown1-5 missing indicators + 0-fill
MarkDowns don't exist before 2011-11-11, and are still missing sometimes after that.
added a missing-indicator column before filling with 0, so the model can tell the difference betweeb true zero and missing.

In [4]:
markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]

for df in [train_merged, test_merged]:
    for col in markdown_cols:
        df[f"{col}_was_missing"] = df[col].isna().astype(int)
    df[markdown_cols] = df[markdown_cols].fillna(0)

## 3. Verification Merge and Missing Value Handling Results

In [5]:
print("TRAIN MERGED:")
print("Shape:", train_merged.shape)
print("\nColumns:", train_merged.columns.tolist())
print("\nHead:")
display(train_merged.head())

TRAIN MERGED:
Shape: (421570, 21)

Columns: ['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Type', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']

Head:


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,...,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,MarkDown1_was_missing,MarkDown2_was_missing,MarkDown3_was_missing,MarkDown4_was_missing,MarkDown5_was_missing
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,0.0,...,0.0,0.0,0.0,211.096358,8.106,1,1,1,1,1
143,1,2,2010-02-05,50605.27,False,A,151315,42.31,2.572,0.0,...,0.0,0.0,0.0,211.096358,8.106,1,1,1,1,1
286,1,3,2010-02-05,13740.12,False,A,151315,42.31,2.572,0.0,...,0.0,0.0,0.0,211.096358,8.106,1,1,1,1,1
429,1,4,2010-02-05,39954.04,False,A,151315,42.31,2.572,0.0,...,0.0,0.0,0.0,211.096358,8.106,1,1,1,1,1
572,1,5,2010-02-05,32229.38,False,A,151315,42.31,2.572,0.0,...,0.0,0.0,0.0,211.096358,8.106,1,1,1,1,1


In [6]:
print("TEST MERGED")
print("Shape:", test_merged.shape)
print("\nColumns:", test_merged.columns.tolist())
display(test_merged.head())

TEST MERGED
Shape: (115064, 20)

Columns: ['Store', 'Dept', 'Date', 'IsHoliday', 'Type', 'Size', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'MarkDown1_was_missing', 'MarkDown2_was_missing', 'MarkDown3_was_missing', 'MarkDown4_was_missing', 'MarkDown5_was_missing']


,Store,Dept,Date,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,MarkDown1_was_missing,MarkDown2_was_missing,MarkDown3_was_missing,MarkDown4_was_missing,MarkDown5_was_missing
0,1,1,2012-11-02,False,A,151315,55.32,3.386,6766.44,5147.7,50.82,3639.9,2737.42,223.462779,6.573,0,0,0,0,0
39,1,2,2012-11-02,False,A,151315,55.32,3.386,6766.44,5147.7,50.82,3639.9,2737.42,223.462779,6.573,0,0,0,0,0
78,1,3,2012-11-02,False,A,151315,55.32,3.386,6766.44,5147.7,50.82,3639.9,2737.42,223.462779,6.573,0,0,0,0,0
117,1,4,2012-11-02,False,A,151315,55.32,3.386,6766.44,5147.7,50.82,3639.9,2737.42,223.462779,6.573,0,0,0,0,0
156,1,5,2012-11-02,False,A,151315,55.32,3.386,6766.44,5147.7,50.82,3639.9,2737.42,223.462779,6.573,0,0,0,0,0


In [7]:
print("Missing values in train_merged:")
display(train_merged.isna().sum()[train_merged.isna().sum() > 0].to_frame("missing_count"))

print("\nMissing values in test_merged:")
display(test_merged.isna().sum()[test_merged.isna().sum() > 0].to_frame("missing_count"))

Missing values in train_merged:


,missing_count



Missing values in test_merged:


,missing_count


In [8]:
sample_store = test_merged["Store"].iloc[0]
sample_check = test_merged.loc[
    test_merged["Store"] == sample_store, ["Store", "Date", "CPI", "Unemployment"]
].sort_values("Date")

print(f"Store {sample_store} last 10 weeks after ffill:")
display(sample_check.tail(10))

Store 1 last 10 weeks after ffill:


,Store,Date,CPI,Unemployment
623,1,2013-07-26,225.17016,6.314
1045,1,2013-07-26,225.17016,6.314
967,1,2013-07-26,225.17016,6.314
928,1,2013-07-26,225.17016,6.314
889,1,2013-07-26,225.17016,6.314
850,1,2013-07-26,225.17016,6.314
811,1,2013-07-26,225.17016,6.314
772,1,2013-07-26,225.17016,6.314
1006,1,2013-07-26,225.17016,6.314
2782,1,2013-07-26,225.17016,6.314


In [9]:
indicator_cols = [f"{c}_was_missing" for c in markdown_cols]

print("MarkDown missing indicator sums:")
display(train_merged[indicator_cols].sum().to_frame("num_missing_rows"))

print("\nNaN in MarkDown cols (should be 0):", train_merged[markdown_cols].isna().sum().sum())

MarkDown missing indicator sums:


,num_missing_rows
MarkDown1_was_missing,270889
MarkDown2_was_missing,310322
MarkDown3_was_missing,284479
MarkDown4_was_missing,286603
MarkDown5_was_missing,270138



NaN in MarkDown cols (should be 0): 0


In [10]:
final_check = pd.DataFrame({
    "check": [
        "train row count preserved",
        "test row count preserved",
        "no NaN in CPI (train/test)",
        "no NaN in Unemployment (train/test)",
        "no NaN in MarkDown cols (train/test)",
    ],
    "passed": [
        len(train_merged) == len(train),
        len(test_merged) == len(test),
        train_merged["CPI"].isna().sum() == 0 and test_merged["CPI"].isna().sum() == 0,
        train_merged["Unemployment"].isna().sum() == 0 and test_merged["Unemployment"].isna().sum() == 0,
        train_merged[markdown_cols].isna().sum().sum() == 0 and test_merged[markdown_cols].isna().sum().sum() == 0,
    ]
})
display(final_check)

,check,passed
0,train row count preserved,True
1,test row count preserved,True
2,no NaN in CPI (train/test),True
3,no NaN in Unemployment (train/test),True
4,no NaN in MarkDown cols (train/test),True


## 4. Calendar and Cyclical Features

In [11]:
for df in [train_merged, test_merged]:
    df["Year"] = df["Date"].dt.year
    df["Month"] = df["Date"].dt.month
    df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)

In [12]:
for df in [train_merged, test_merged]:
    df["Week_sin"] = np.sin(2 * np.pi * df["WeekOfYear"] / 52)
    df["Week_cos"] = np.cos(2 * np.pi * df["WeekOfYear"] / 52)

## 5. Holiday Features

In [13]:
holiday_dates = {
    "SuperBowl": ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"],
    "LaborDay": ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"],
    "Thanksgiving": ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"],
    "Christmas": ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"],
}

for df in [train_merged, test_merged]:
    for holiday_name, dates in holiday_dates.items():
        df[f"Is{holiday_name}"] = df["Date"].isin(pd.to_datetime(dates)).astype(int)

## 6. Verification Calendar and Holiday Features

In [14]:
print("Sample of calendar features:")
display(train_merged[["Date", "Year", "Month", "WeekOfYear", "Week_sin", "Week_cos"]].head())

print("\nWeek_sin/cos range check (should be between -1 and 1):")
print(train_merged[["Week_sin", "Week_cos"]].describe())

print("\nHoliday flag counts (train):")
holiday_cols = ["IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas"]
display(train_merged[holiday_cols].sum().to_frame("num_rows"))

print("Original IsHoliday=True count:", train_merged["IsHoliday"].sum())
print("Sum of our 4 holiday flags:", train_merged[holiday_cols].sum().sum())

Sample of calendar features:


,Date,Year,Month,WeekOfYear,Week_sin,Week_cos
0,2010-02-05,2010,2,5,0.568065,0.822984
143,2010-02-05,2010,2,5,0.568065,0.822984
286,2010-02-05,2010,2,5,0.568065,0.822984
429,2010-02-05,2010,2,5,0.568065,0.822984
572,2010-02-05,2010,2,5,0.568065,0.822984



Week_sin/cos range check (should be between -1 and 1):
           Week_sin       Week_cos
count  4.215700e+05  421570.000000
mean   2.011720e-02      -0.076954
std    7.250491e-01       0.684090
min   -1.000000e+00      -1.000000
25%   -7.485107e-01      -0.748511
50%    6.432491e-16      -0.120537
75%    7.485107e-01       0.568065
max    1.000000e+00       1.000000

Holiday flag counts (train):


,num_rows
IsSuperBowl,8895
IsLaborDay,8861
IsThanksgiving,5959
IsChristmas,5946


Original IsHoliday=True count: 29661
Sum of our 4 holiday flags: 29661


## 7. Store Encoding (Type, Size)

In [15]:
# Label encoding tree-based მოდელებისთვის (LightGBM, XGBoost)
type_mapping = {"A": 0, "B": 1, "C": 2}

for df in [train_merged, test_merged]:
    df["Type_encoded"] = df["Type"].map(type_mapping)

# One-hot encoding non-tree მოდელებისთვის (linear, neural და ა.შ.)
train_type_dummies = pd.get_dummies(train_merged["Type"], prefix="Type")
test_type_dummies = pd.get_dummies(test_merged["Type"], prefix="Type")

train_merged = pd.concat([train_merged, train_type_dummies], axis=1)
test_merged = pd.concat([test_merged, test_type_dummies], axis=1)

# დარწმუნება, რომ train/test-ს ერთნაირი one-hot სვეტები აქვს
assert set(train_type_dummies.columns) == set(test_type_dummies.columns), \
    "Type one-hot columns arent the same in train and test"

# True/False - 1/0
onehot_cols = ["Type_A", "Type_B", "Type_C"]
train_merged[onehot_cols] = train_merged[onehot_cols].astype(int)
test_merged[onehot_cols] = test_merged[onehot_cols].astype(int)

## 8. MarkDown Aggregated Features

In [16]:
for df in [train_merged, test_merged]:
    df["markdown_available_period"] = (df["Date"] >= pd.Timestamp("2011-11-11")).astype(int)
    df["markdown_missing_count"] = df[[f"{c}_was_missing" for c in markdown_cols]].sum(axis=1)

    df["total_markdown"] = df[markdown_cols].sum(axis=1)
    df["abs_total_markdown"] = df[markdown_cols].abs().sum(axis=1)
    df["positive_markdown_sum"] = df[markdown_cols].clip(lower=0).sum(axis=1)
    df["negative_markdown_sum"] = df[markdown_cols].clip(upper=0).sum(axis=1)
    df["has_markdown_signal"] = (df["abs_total_markdown"] > 0).astype(int)

## 9. Verification Store Encoding and MarkDown Aggregated Features

In [17]:
print("Type encoding check:")
display(train_merged[["Type", "Type_encoded"]].drop_duplicates())

print("\nMarkDown aggregated features sample:")
display(train_merged[markdown_cols + ["total_markdown", "abs_total_markdown",
                                        "positive_markdown_sum", "negative_markdown_sum"]].describe())

print("Count where negative_markdown_sum < 0:", (train_merged["negative_markdown_sum"] < 0).sum())
print("Count where positive_markdown_sum > 0:", (train_merged["positive_markdown_sum"] > 0).sum())

print("\nmarkdown_available_period check (should be 0 before 2011-11-11, 1 after):")
display(
    train_merged.groupby("markdown_available_period")["Date"].agg(["min", "max"])
)

print("\nType one-hot columns:", [c for c in train_merged.columns if c.startswith("Type_")])
display(train_merged[["Type", "Type_A", "Type_B", "Type_C"]].drop_duplicates())

Type encoding check:


,Type,Type_encoded
0,A,0
20482,B,1
286548,C,2



MarkDown aggregated features sample:


,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,total_markdown,abs_total_markdown,positive_markdown_sum,negative_markdown_sum
count,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000
mean,2590.074819,879.974298,468.087665,1083.132268,1662.772385,6684.041435,6684.243915,6684.142675,-0.101240
std,6052.385934,5084.538801,5528.873453,3894.529945,4207.629321,14750.941551,14750.975434,14750.957865,4.303235
min,0.000000,-265.760000,-29.100000,0.000000,0.000000,0.000000,0.000000,0.000000,-265.760000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2809.050000,2.200000,4.540000,425.290000,2168.040000,8075.260000,8075.260000,8075.260000,0.000000
max,88646.760000,104519.540000,141630.610000,67474.850000,108519.280000,160510.610000,160510.610000,160510.610000,0.000000


Count where negative_markdown_sum < 0: 1568
Count where positive_markdown_sum > 0: 151432

markdown_available_period check (should be 0 before 2011-11-11, 1 after):


,min,max
markdown_available_period,,
0,2010-02-05,2011-11-04
1,2011-11-11,2012-10-26



Type one-hot columns: ['Type_encoded', 'Type_A', 'Type_B', 'Type_C']


,Type,Type_A,Type_B,Type_C
0,A,1,0,0
20482,B,0,1,0
286548,C,0,0,1


## 10. Lag and Rolling Features (Store-Dept level)

Lag/rolling features are computed on for Store-Dept group, so that early test weeks can still pull history from train. Store-Dept pairs with no train history get a Dept-level fallback.

In [18]:
train_for_lag = train_merged[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
train_for_lag["is_train"] = 1

test_for_lag = test_merged[["Store", "Dept", "Date"]].copy()
test_for_lag["Weekly_Sales"] = np.nan
test_for_lag["is_train"] = 0

combined = pd.concat([train_for_lag, test_for_lag], axis=0)
combined = combined.sort_values(["Store", "Dept", "Date"])

In [19]:
lag_weeks = [1, 4, 52]

for lag in lag_weeks:
    combined[f"Sales_lag_{lag}"] = combined.groupby(["Store", "Dept"])["Weekly_Sales"].shift(lag)

In [20]:
combined["Sales_shifted"] = combined.groupby(["Store", "Dept"])["Weekly_Sales"].shift(1)

combined["Sales_roll_mean_4"] = (
    combined.groupby(["Store", "Dept"])["Sales_shifted"]
    .transform(lambda s: s.rolling(4, min_periods=1).mean())
)
combined["Sales_roll_std_4"] = (
    combined.groupby(["Store", "Dept"])["Sales_shifted"]
    .transform(lambda s: s.rolling(4, min_periods=1).std())
)
combined["Sales_roll_mean_12"] = (
    combined.groupby(["Store", "Dept"])["Sales_shifted"]
    .transform(lambda s: s.rolling(12, min_periods=1).mean())
)

In [21]:
lag_roll_cols = [f"Sales_lag_{l}" for l in lag_weeks] + ["Sales_roll_mean_4", "Sales_roll_std_4", "Sales_roll_mean_12"]

train_lag_features = combined.loc[combined["is_train"] == 1, ["Store", "Dept", "Date"] + lag_roll_cols]
test_lag_features = combined.loc[combined["is_train"] == 0, ["Store", "Dept", "Date"] + lag_roll_cols]

train_merged = train_merged.merge(train_lag_features, on=["Store", "Dept", "Date"], how="left")
test_merged = test_merged.merge(test_lag_features, on=["Store", "Dept", "Date"], how="left")

assert len(train_merged) == len(train)
assert len(test_merged) == len(test)

### 10.1 Fallback for Store-Dept pairs with no train history

In [22]:
dept_avg_sales = train_merged.groupby("Dept")["Weekly_Sales"].mean()

mean_fallback_cols = [f"Sales_lag_{l}" for l in lag_weeks] + ["Sales_roll_mean_4", "Sales_roll_mean_12"]

for col in mean_fallback_cols:
    train_merged[col] = train_merged[col].fillna(train_merged["Dept"].map(dept_avg_sales))
    test_merged[col] = test_merged[col].fillna(test_merged["Dept"].map(dept_avg_sales))

# სტანდარტული გადახრისთვის fallback 0-ია
train_merged["Sales_roll_std_4"] = train_merged["Sales_roll_std_4"].fillna(0)
test_merged["Sales_roll_std_4"] = test_merged["Sales_roll_std_4"].fillna(0)

## 11. Final Verification

In [23]:
new_feature_cols = [f"Sales_lag_{l}" for l in lag_weeks] + ["Sales_roll_mean_4", "Sales_roll_std_4", "Sales_roll_mean_12"]

print("Remaining NaN in lag/rolling features (should be 0 after fallback):")
display(train_merged[new_feature_cols].isna().sum().to_frame("missing"))
display(test_merged[new_feature_cols].isna().sum().to_frame("missing"))

print("\nSample for one Store-Dept with full history:")
display(
    train_merged.loc[(train_merged["Store"] == 1) & (train_merged["Dept"] == 1),
                      ["Date", "Weekly_Sales"] + new_feature_cols].head(10)
)

print("\nSanity: final combined shape check")
print("train_merged:", train_merged.shape)
print("test_merged:", test_merged.shape)
assert train_merged[new_feature_cols].isna().sum().sum() == 0
assert test_merged[new_feature_cols].isna().sum().sum() == 0

Remaining NaN in lag/rolling features (should be 0 after fallback):


,missing
Sales_lag_1,0
Sales_lag_4,0
Sales_lag_52,0
Sales_roll_mean_4,0
Sales_roll_std_4,0
Sales_roll_mean_12,0


,missing
Sales_lag_1,0
Sales_lag_4,0
Sales_lag_52,0
Sales_roll_mean_4,0
Sales_roll_std_4,0
Sales_roll_mean_12,0



Sample for one Store-Dept with full history:


,Date,Weekly_Sales,Sales_lag_1,Sales_lag_4,Sales_lag_52,Sales_roll_mean_4,Sales_roll_std_4,Sales_roll_mean_12
0,2010-02-05,24924.50,19213.485088,19213.485088,19213.485088,19213.485088,0.000000,19213.485088
73,2010-02-12,46039.49,24924.500000,19213.485088,19213.485088,24924.500000,0.000000,24924.500000
145,2010-02-19,41595.55,46039.490000,19213.485088,19213.485088,35481.995000,14930.552614,35481.995000
218,2010-02-26,19403.54,41595.550000,19213.485088,19213.485088,37519.846667,11131.900957,37519.846667
290,2010-03-05,21827.90,19403.540000,24924.500000,19213.485088,32990.770000,12832.106391,32990.770000
363,2010-03-12,21043.39,21827.900000,46039.490000,19213.485088,32216.620000,13554.047185,30758.196000
436,2010-03-19,22136.64,21043.390000,41595.550000,19213.485088,25967.595000,10467.484020,29139.061667
508,2010-03-26,26229.21,22136.640000,19403.540000,19213.485088,21102.867500,1222.784968,28138.715714
580,2010-04-02,57258.43,26229.210000,21827.900000,19213.485088,22809.285000,2325.929203,27900.027500
652,2010-04-09,42960.91,57258.430000,21043.390000,19213.485088,31666.917500,17206.391261,31162.072222



Sanity: final combined shape check
train_merged: (421570, 47)
test_merged: (115064, 46)


## 12. MarkDown Raw Columns (NaN preserved) — for tree-model native missing handling

with the 0-filled MarkDown columns, we keep a "raw" version with NaN preserved, so tree-based models (LightGBM/XGBoost) can be tested both ways: 0-fill+indicator vs own NaN handling.

In [24]:
for df in [train_merged, test_merged]:
    for col in markdown_cols:
        df[f"{col}_raw"] = df[col].where(df[f"{col}_was_missing"] == 0, np.nan)

In [25]:
print("Raw MarkDown NaN counts (should match original missing counts):")
display(train_merged[[f"{c}_raw" for c in markdown_cols]].isna().sum().to_frame("nan_count"))

display(train_merged[indicator_cols].sum().to_frame("indicator_sum"))

Raw MarkDown NaN counts (should match original missing counts):


,nan_count
MarkDown1_raw,270889
MarkDown2_raw,310322
MarkDown3_raw,284479
MarkDown4_raw,286603
MarkDown5_raw,270138


,indicator_sum
MarkDown1_was_missing,270889
MarkDown2_was_missing,310322
MarkDown3_was_missing,284479
MarkDown4_was_missing,286603
MarkDown5_was_missing,270138


## 13. CPI/Unemployment Imputation Comparison (Validation Simulation)

In [26]:
validation_cutoff = train_merged["Date"].max() - pd.Timedelta(weeks=13)

sim_train = train_merged[train_merged["Date"] <= validation_cutoff].copy()
sim_val = train_merged[train_merged["Date"] > validation_cutoff].copy()

print("Sim train date range:", sim_train["Date"].min(), "-", sim_train["Date"].max())
print("Sim val date range:", sim_val["Date"].min(), "-", sim_val["Date"].max())

Sim train date range: 2010-02-05 00:00:00 - 2012-07-27 00:00:00
Sim val date range: 2012-08-03 00:00:00 - 2012-10-26 00:00:00


In [27]:
sim_val_masked = sim_val.copy()
sim_val_masked[["CPI", "Unemployment"]] = np.nan

In [28]:
def impute_ffill(sim_train, sim_val_masked):
    combined_sim = pd.concat([sim_train, sim_val_masked], axis=0).sort_values(["Store", "Date"])
    for col in ["CPI", "Unemployment"]:
        combined_sim[col] = combined_sim.groupby("Store")[col].ffill()
    return combined_sim.loc[combined_sim["Date"] > validation_cutoff]

val_ffill = impute_ffill(sim_train, sim_val_masked)

In [29]:
def impute_extrapolate(sim_train, sim_val_masked):
    combined_sim = pd.concat([sim_train, sim_val_masked], axis=0).sort_values(["Store", "Date"])
    for col in ["CPI", "Unemployment"]:
        combined_sim[col] = combined_sim.groupby("Store")[col].transform(
            lambda s: s.interpolate(method="linear", limit_direction="forward")
        )
    return combined_sim.loc[combined_sim["Date"] > validation_cutoff]

val_extrapolate = impute_extrapolate(sim_train, sim_val_masked)

In [30]:
val_drop = sim_val.copy()  # CPI/Unemployment columns excluded from features later

In [31]:
from sklearn.ensemble import RandomForestRegressor

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

def evaluate_method(train_df, val_df, feature_cols, label):
    X_train = train_df[feature_cols].fillna(0)
    y_train = train_df["Weekly_Sales"]
    X_val = val_df[feature_cols].fillna(0)
    y_val = val_df["Weekly_Sales"]

    model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    score = wmae(y_val.values, preds, val_df["IsHoliday"].values)
    print(f"{label}: WMAE = {score:.2f}")
    return score

base_features = ["Store", "Dept", "Size", "Temperature", "Fuel_Price", "Type_encoded", "WeekOfYear"]

score_ffill = evaluate_method(sim_train, val_ffill, base_features + ["CPI", "Unemployment"], "ffill")
score_extrapolate = evaluate_method(sim_train, val_extrapolate, base_features + ["CPI", "Unemployment"], "extrapolation")
score_drop = evaluate_method(sim_train, val_drop, base_features, "drop (no CPI/Unemployment)")

ffill: WMAE = 5402.96
extrapolation: WMAE = 5402.96
drop (no CPI/Unemployment): WMAE = 5458.03
